# Bert-base for CCF 

## Importing Libraries

In [16]:
import numpy as np
import torch
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader
import logging
import torch
import torch.nn as nn
import numpy as np
from data.preprocessing import TransactionDataset
from torch.utils.data.sampler import WeightedRandomSampler
from torch.utils.data import DataLoader
from data.preprocessing import random_split_dataset
import torch.optim as optim
import torch.nn.functional as F
from transformers import BertForSequenceClassification
import torch
import os
import onnx
import onnxruntime as ort
np.random.seed(42)
device = torch.device('cpu')

In [17]:
logger = logging.getLogger()
logger.setLevel(logging.INFO)
# formatter
formatter = logging.Formatter(
    '%(asctime)s - %(levelname)s(%(name)s): %(message)s')
# console handler
consH = logging.StreamHandler()
consH.setFormatter(formatter)
consH.setLevel(logging.INFO)
# add the handlers to the logger
logger.addHandler(consH)
# File handler
request_file_handler = True
log = logger

## Data Preperation

In [18]:
def class_imbalance_sampler(labels):
    class_count = torch.bincount(labels)
    class_weighting = 1. / class_count.float()
    sample_weights = class_weighting[labels]
    sampler = WeightedRandomSampler(sample_weights, len(labels))
    return sampler

Preprocessing steps

1. Followed the pre-preprocessing steps mentioned in the TabFormer paper, where the sequences are generated by combining the 10 transactions of a user as one sequence.  
2. Each column is encoded by applying various encoding technqiues depending on it's datatype.

In [19]:
dataset = TransactionDataset(root='./CCF_data/',
                                fname='train_set_all_fraud',
                                fextension='',
                                vocab_dir='op_1',
                                return_labels=True,
                                )

vocab = dataset.vocab
custom_special_tokens = vocab.get_special_tokens()

# split dataset into train, val, test [0.6. 0.2, 0.2]
totalN = len(dataset)
trainN = int(0.6 * totalN)

valtestN = totalN - trainN
valN = int(valtestN * 0.5)
testN = valtestN - valN

assert totalN == trainN + valN + testN

lengths = [trainN, valN, testN]

log.info(f"# lengths: train [{trainN}]  valid [{valN}]  test [{testN}]")
log.info("# lengths: train [{:.2f}]  valid [{:.2f}]  test [{:.2f}]".format(trainN / totalN, valN / totalN, testN / totalN))

train_dataset, eval_dataset, test_dataset = random_split_dataset(dataset, lengths)

window_label = torch.tensor(dataset.window_label, dtype=torch.long)
window_label_train = window_label[train_dataset.indices]

train_dataloader = DataLoader(train_dataset, batch_size=128, num_workers=5, drop_last=False, sampler=class_imbalance_sampler(window_label_train))
eval_dataloader = DataLoader(eval_dataset, batch_size=128, num_workers=5, drop_last=False, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=128, num_workers=5, drop_last=False, shuffle=False)

INFO:data.preprocessing:cached encoded data is read from train_set_all_fraud.encoded.csv
2025-01-15 17:15:48,893 - INFO(data.preprocessing): cached encoded data is read from train_set_all_fraud.encoded.csv
INFO:data.preprocessing:read data : (2394164, 12)
2025-01-15 17:15:49,589 - INFO(data.preprocessing): read data : (2394164, 12)


{0.0: 239417, 2.0: 239417, 3.0: 239417, 6.0: 239417, 9.0: 239417, 4.0: 239416, 5.0: 239416, 7.0: 239416, 8.0: 239416, 1.0: 239415}


INFO:data.preprocessing:total columns: ['User', 'Card', 'Timestamp', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'Is Fraud?']
2025-01-15 17:15:49,866 - INFO(data.preprocessing): total columns: ['User', 'Card', 'Timestamp', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'Is Fraud?']
INFO:data.preprocessing:total vocabulary size: 83241
2025-01-15 17:15:49,867 - INFO(data.preprocessing): total vocabulary size: 83241
INFO:data.preprocessing:column : User, vocab size : 2000
2025-01-15 17:15:49,867 - INFO(data.preprocessing): column : User, vocab size : 2000
INFO:data.preprocessing:column : Card, vocab size : 9
2025-01-15 17:15:49,869 - INFO(data.preprocessing): column : Card, vocab size : 9
INFO:data.preprocessing:column : Timestamp, vocab size : 10
2025-01-15 17:15:49,869 - INFO(data.preprocessing): column : Timestamp, vocab size : 10
INFO:data.preprocessing:column : Amount, vocab siz

In [20]:
len(vocab)

83241

In [21]:
len(train_dataloader)

2232

## Defining the model

Loading the pre-trained bert model

In [ ]:
# Load pre-trained BERT model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2,hidden_dropout_prob=0.3,vocab_size=len(vocab),ignore_mismatched_sizes=True)  # Binary classification

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized because the shapes did not match:
- bert.embeddings.word_embeddings.weight: found shape torch.Size([30522, 768]) in the checkpoint and torch.Size([83241, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83241, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.3, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

Optimizer

In [37]:
from transformers import AdamW
optimizer = optim.Adam(model.parameters(),lr=2e-5, weight_decay=1e-2)

Focal loss works well in case of imbalanced datasets. Focal loss focuses on the examples that the model gets wrong rather than the ones that it can confidently predict, ensuring that predictions on hard examples improve over time rather than becoming overly confident with easy ones.

https://towardsdatascience.com/focal-loss-a-better-alternative-for-cross-entropy-1d073d92d075

In [38]:
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce_loss = torch.nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, targets):
        ce_loss = self.ce_loss(logits, targets)
        pt = torch.exp(-ce_loss)  # Probability of the correct class
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.alpha is not None:
            focal_loss = self.alpha[targets] * focal_loss
        return focal_loss.mean()

alpha = torch.tensor([0.1, 0.7])  
alpha = alpha.to(device)

criterion = FocalLoss(alpha=alpha, gamma=1.5)

## Model Training

Epoch 1

In [39]:
for epoch in range(1):  
    model.train()
    total_train_loss = 0.0  
    all_train_preds=[]
    all_train_labels=[]
    iter=0

    for batch in train_dataloader:
        
        input_ids, labels = batch 
        input_ids, labels = input_ids.to(device), labels.to(device)
        #break
        labels = labels[:, -1]
        
        attention_mask = torch.ones_like(input_ids).long()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        print(f"Iter {iter}, Loss: {loss.item()}")
        iter+=1
        total_train_loss += loss.item()

        # probs = torch.nn.functional.softmax(logits, dim=-1)
        
        # preds = (probs[:, 1] > 0.5).long()
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())
        print(classification_report(all_train_labels, all_train_preds, target_names=["Non-Fraud", "Fraud"]))
        if iter % 500 == 0:
            torch.save(model.state_dict(), f"alpha_00_07/iter{iter}.pt")
    all_train_preds = np.array(all_train_preds).astype(int)
    all_train_labels = np.array(all_train_labels).astype(int)
    torch.save(model.state_dict(), f"alpha_00_07/iter{iter}.pt")
    print(f"\nEpoch {epoch}")
    print(f"Average Training Loss: {total_train_loss / len(train_dataloader):.4f}")
    print("Overall Training Classification Report:")
    print(classification_report(all_train_labels, all_train_preds, target_names=["Non-Fraud", "Fraud"]))

Iter 0, Loss: 0.04208042472600937
              precision    recall  f1-score   support

   Non-Fraud       0.92      0.84      0.88       114
       Fraud       0.25      0.43      0.32        14

    accuracy                           0.80       128
   macro avg       0.59      0.64      0.60       128
weighted avg       0.85      0.80      0.82       128

Iter 1, Loss: 0.05422700196504593
              precision    recall  f1-score   support

   Non-Fraud       0.89      0.68      0.77       220
       Fraud       0.20      0.50      0.29        36

    accuracy                           0.65       256
   macro avg       0.55      0.59      0.53       256
weighted avg       0.80      0.65      0.70       256

Iter 2, Loss: 0.039868202060461044
              precision    recall  f1-score   support

   Non-Fraud       0.91      0.57      0.70       339
       Fraud       0.15      0.56      0.23        45

    accuracy                           0.57       384
   macro avg       0.53  

Epoch 2

In [42]:
for epoch in range(1):  
    model.train()
    total_train_loss = 0.0  
    all_train_preds=[]
    all_train_labels=[]
    iter=0

    for batch in train_dataloader:
        
        input_ids, labels = batch 
        input_ids, labels = input_ids.to(device), labels.to(device)
        #break
        labels = labels[:, -1]
        
        attention_mask = torch.ones_like(input_ids).long()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        print(f"Iter {iter}, Loss: {loss.item()}")
        iter+=1
        total_train_loss += loss.item()

        # probs = torch.nn.functional.softmax(logits, dim=-1)
        
        # preds = (probs[:, 1] > 0.5).long()
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())
        print(classification_report(all_train_labels, all_train_preds, target_names=["Non-Fraud", "Fraud"]))
        if iter % 500 == 0:
            torch.save(model.state_dict(), f"alpha_00_07_epoch1/iter{iter}.pt")
    all_train_preds = np.array(all_train_preds).astype(int)
    all_train_labels = np.array(all_train_labels).astype(int)
    torch.save(model.state_dict(), f"alpha_00_07_epoch1/iter{iter}.pt")
    print(f"\nEpoch {epoch}")
    print(f"Average Training Loss: {total_train_loss / len(train_dataloader):.4f}")
    print("Overall Training Classification Report:")
    print(classification_report(all_train_labels, all_train_preds, target_names=["Non-Fraud", "Fraud"]))

Iter 0, Loss: 0.019658781588077545
              precision    recall  f1-score   support

   Non-Fraud       1.00      0.67      0.80       107
       Fraud       0.38      1.00      0.55        21

    accuracy                           0.73       128
   macro avg       0.69      0.84      0.67       128
weighted avg       0.90      0.73      0.76       128

Iter 1, Loss: 0.015580553561449051
              precision    recall  f1-score   support

   Non-Fraud       1.00      0.74      0.85       210
       Fraud       0.46      1.00      0.63        46

    accuracy                           0.79       256
   macro avg       0.73      0.87      0.74       256
weighted avg       0.90      0.79      0.81       256

Iter 2, Loss: 0.01198592223227024
              precision    recall  f1-score   support

   Non-Fraud       1.00      0.79      0.88       315
       Fraud       0.51      1.00      0.67        69

    accuracy                           0.83       384
   macro avg       0.75 

## Model Validation

Epoch-1 Validation

In [40]:
from sklearn.metrics import classification_report, accuracy_score

# Validation loop
model.eval()  
all_preds = []
all_labels = []

with torch.no_grad():  
    for batch in eval_dataloader:  
        input_ids, labels = batch
        input_ids, labels = input_ids.to(device), labels.to(device)

        labels = labels[:, -1]

        attention_mask = torch.ones_like(input_ids).long()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  

        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        #preds = (probs[:, 1] > 0.5).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=["Non-Fraud", "Fraud"]))

accuracy = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {accuracy:.4f}")

Classification Report:
              precision    recall  f1-score   support

   Non-Fraud       1.00      0.89      0.94     94428
       Fraud       0.07      0.93      0.13       783

    accuracy                           0.89     95211
   macro avg       0.53      0.91      0.54     95211
weighted avg       0.99      0.89      0.94     95211

Validation Accuracy: 0.8943


Epoch -2 validation

In [43]:
from sklearn.metrics import classification_report, accuracy_score

# Validation loop
model.eval()  
all_preds = []
all_labels = []

with torch.no_grad():  
    for batch in eval_dataloader:  
        input_ids, labels = batch
        input_ids, labels = input_ids.to(device), labels.to(device)

        labels = labels[:, -1]

        attention_mask = torch.ones_like(input_ids).long()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  

        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        #preds = (probs[:, 1] > 0.5).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=["Non-Fraud", "Fraud"]))

accuracy = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {accuracy:.4f}")

Classification Report:
              precision    recall  f1-score   support

   Non-Fraud       1.00      0.99      1.00     94428
       Fraud       0.48      0.83      0.61       783

    accuracy                           0.99     95211
   macro avg       0.74      0.91      0.80     95211
weighted avg       0.99      0.99      0.99     95211

Validation Accuracy: 0.9912


## Loading the model checkpoint for inferencing

In [60]:
#model_load = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model_load=BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2,hidden_dropout_prob=0.3,vocab_size=len(vocab),ignore_mismatched_sizes=True)  # Binary classification

model_load.load_state_dict(torch.load("alpha_00_07_epoch1/iter2232.pt"))
model_load.to(device)

print("Model loaded successfully!")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized because the shapes did not match:
- bert.embeddings.word_embeddings.weight: found shape torch.Size([30522, 768]) in the checkpoint and torch.Size([83241, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/ck/h8vxl6cx4wb56krbyr57sf1c0000gn/T/ipykernel_23787/1954346680.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pick

Model loaded successfully!


## Quantization

In [61]:
device = torch.device("cpu")
model_load.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83241, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.3, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [49]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    print('Size (MB):', os.path.getsize("temp.p")/(1024*1024))
    os.remove('temp.p')

In [50]:
print_size_of_model(model_load)

Size (MB): 572.1600275039673


In [62]:
quantized_model = torch.quantization.quantize_dynamic(
    model_load, {torch.nn.Linear}, dtype=torch.qint8
)

In [63]:
print_size_of_model(quantized_model)

Size (MB): 327.5210771560669


In [94]:
torch.save(quantized_model.state_dict(), f"alpha_00_07_epoch1/quantized_iter{iter}.pt")

Validating the quantized model

In [72]:
from sklearn.metrics import classification_report, accuracy_score

# Validation loop
quantized_model.eval()  
all_preds = []
all_labels = []

with torch.no_grad():  
    for batch in eval_dataloader:  
        input_ids, labels = batch
        input_ids, labels = input_ids.to(device), labels.to(device)

        labels = labels[:, -1]

        attention_mask = torch.ones_like(input_ids).long()

        outputs = quantized_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  

        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        #preds = (probs[:, 1] > 0.5).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=["Non-Fraud", "Fraud"]))

accuracy = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {accuracy:.4f}")

Classification Report:
              precision    recall  f1-score   support

   Non-Fraud       1.00      0.98      0.99     94428
       Fraud       0.26      0.88      0.40       783

    accuracy                           0.98     95211
   macro avg       0.63      0.93      0.69     95211
weighted avg       0.99      0.98      0.98     95211

Validation Accuracy: 0.9780


## ONNX Conversion

In [22]:
for batch in train_dataloader:
    input_ids, labels = batch 
    print(input_ids[0].shape)
    break

torch.Size([120])


In [23]:
data_onnx = input_ids[0].unsqueeze(0).to(device)

In [24]:
data_onnx.shape

torch.Size([1, 120])

In [26]:
data_onnx

tensor([[ 1046,  2012,  2020,  2029,  2037,  2044, 49621, 60621, 62104, 83111,
         83218,     1,  1046,  2008,  2020,  2028,  2037,  2040, 49621, 60621,
         62365, 83110, 83218,     1,  1046,  2012,  2020,  2033,  2037,  2041,
         49500, 60628, 62454, 83111, 83218,     1,  1046,  2012,  2020,  2035,
          2037,  2041, 49500, 60628, 62454, 83111, 83218,     1,  1046,  2008,
          2020,  2031,  2037,  2041, 49500, 60628, 62454, 83111, 83218,     1,
          1046,  2011,  2020,  2030,  2036,  2080, 49621, 60621, 64635, 83125,
         83218,     1,  1046,  2009,  2020,  2033,  2037, 13326, 49621, 60621,
         62104, 83151, 83218,     1,  1046,  2011,  2020,  2035,  2037,  2040,
         49621, 60621, 62365, 83110, 83218,     1,  1046,  2012,  2020,  2027,
          2036,  3367, 49621, 60621, 63568, 83127, 83218,     1,  1046,  2008,
          2020,  2034,  2037,  2060, 49621, 60621, 62104, 83131, 83218,     1]])

Converting the model to ONNX

In [80]:
onnx_file_path = "bert_model.onnx"
torch.onnx.export(
    model,             # The model
    data_onnx,                   # Dummy input
    onnx_file_path,                # Path to save the ONNX model
    export_params=True,            # Store the trained parameters in the model file
    opset_version=14,              # ONNX version to use
    do_constant_folding=True,      # Optimize the graph
    input_names=["input_ids"],     # Name of the input layer(s)
    output_names=["output"],       # Name of the output layer(s)
    dynamic_axes={                 # Enable dynamic batching
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "output": {0: "batch_size"}
    }
)

print(f"Model exported to {onnx_file_path}")

Model exported to bert_model.onnx


Importing the ONNX model and infering 

In [92]:
# Load the exported ONNX model
session = ort.InferenceSession("bert_model.onnx")

data_onnx = np.array(data_onnx, dtype=np.int64)  # Ensure dtype matches

#input dictionary
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
input_dict = {input_name: data_onnx}

# Run inference
output = session.run([output_name], input_dict)

# Print the output
print(f"Model output: {output}")

Model output: [array([[ 1.4293313, -1.5043263]], dtype=float32)]


## Quantization of ONNX Model

In [1]:
from onnxruntime.quantization import quantize_dynamic, QuantType

In [11]:
quantize_dynamic(
    model_input="bert_model.onnx",
    model_output="quantized_model_8bit.onnx",
    weight_type=QuantType.QUInt8  # Quantize weights to 8-bit unsigned integers
)

In [7]:
quantized_model = onnx.load("quantized_model.onnx")

Inferencing using quantized bert model

In [27]:
# Load the exported ONNX model
session = ort.InferenceSession("quantized_model.onnx")

data_onnx = np.array(data_onnx, dtype=np.int64)  # Ensure dtype matches

#input dictionary
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
input_dict = {input_name: data_onnx}

# Run inference
output = session.run([output_name], input_dict)

# Print the output
print(f"Model output: {output}")

Model output: [array([[ 0.48917797, -0.37059432]], dtype=float32)]


/var/folders/ck/h8vxl6cx4wb56krbyr57sf1c0000gn/T/ipykernel_20522/155489593.py:4: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  data_onnx = np.array(data_onnx, dtype=np.int64)  # Ensure dtype matches
